# Step 1: CD Overview and Deployment Goal

CD stands for **Continuous Delivery** or **Continuous Deployment**. In this mini MLOps project, CD means preparing the trained model so it can be safely released as a prediction service.

The trained model is saved as a `.joblib` file. In this step, the model is not retrained. It is only prepared for a simple release scenario using a local FastAPI plan.

The selected deployment strategy is **Shadow Deployment**. Shadow Deployment means a new model makes predictions in the background first, without directly affecting real users. This is useful for a pricing-related model because predictions can be monitored before the model is promoted.

This project remains a learning simulation for estimating online transportation fare. It is not a real production pricing system.


In [1]:
from pathlib import Path
from datetime import datetime
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
elif not (project_root / "data").exists():
    project_root = project_root.parent

models_dir = project_root / "models"
reports_dir = project_root / "reports"
diagrams_dir = project_root / "diagrams"
api_dir = project_root / "api"
notebooks_dir = project_root / "notebooks"

for folder in [models_dir, reports_dir, diagrams_dir, api_dir, notebooks_dir]:
    folder.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "baseline_price_model.joblib"
quality_checklist_path = reports_dir / "quality_checklist.md"
model_evaluation_report_path = reports_dir / "model_evaluation_report.md"
registry_path = models_dir / "model_registry.json"
cd_report_path = reports_dir / "cd_scenario_report.md"
pipeline_design_report_path = reports_dir / "mlops_pipeline_design.md"
diagram_md_path = diagrams_dir / "mlops_pipeline_diagram.md"
diagram_png_path = diagrams_dir / "mlops_pipeline_diagram.png"

print("Project root:", project_root)


Project root: d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops-tarif-transportasi-online


In [2]:
model_file_available = model_path.exists()
ci_checklist_available = quality_checklist_path.exists()
ct_checklist_available = quality_checklist_path.exists()
model_performance_available = model_evaluation_report_path.exists()
ready_for_cd_design = all([
    model_file_available,
    ci_checklist_available,
    ct_checklist_available,
    model_performance_available,
])

readiness_summary = pd.DataFrame({
    "Readiness Item": [
        "Model file available",
        "CI checklist available",
        "CT checklist available",
        "Model performance available",
        "Ready for CD scenario design",
    ],
    "Status": [
        "YES" if model_file_available else "NO",
        "YES" if ci_checklist_available else "NO",
        "YES" if ct_checklist_available else "NO",
        "YES" if model_performance_available else "NO",
        "YES" if ready_for_cd_design else "NO",
    ],
    "Path / Note": [
        str(model_path),
        str(quality_checklist_path),
        str(quality_checklist_path),
        str(model_evaluation_report_path),
        "Proceed with local CD design only" if ready_for_cd_design else "Complete earlier steps first",
    ],
})

display(readiness_summary)


,Readiness Item,Status,Path / Note
0,Model file available,YES,d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops...
1,CI checklist available,YES,d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops...
2,CT checklist available,YES,d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops...
3,Model performance available,YES,d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops...
4,Ready for CD scenario design,YES,Proceed with local CD design only


# Step 2: Simple Model Registry Design

A model registry is used to track model versions, metrics, file path, status, and approval notes.

For this mini project, the registry can be a simple JSON file. It does not need advanced tools such as MLflow.

The model should only be marked as approved when CI and CT checks are acceptable. The current model is approved for a shadow deployment simulation with monitoring notes because service-level error still has one warning.


In [3]:
model_metadata = {
    "model_name": "online_transport_fare_estimator",
    "model_version": "v1.0-baseline-random-forest",
    "model_type": "Random Forest Regressor",
    "model_file": "models/baseline_price_model.joblib",
    "target_column": "price",
    "problem_type": "regression",
    "features_used": [
        "distance",
        "cab_type",
        "source",
        "destination",
        "name",
        "hour",
        "day",
        "month",
        "day_of_week",
    ],
    "features_excluded": {
        "id": "identifier",
        "product_id": "identifier or product code",
        "price": "target column",
        "surge_multiplier": "closely related to the pricing mechanism",
        "weather_columns": "not merged yet because time and location alignment is not prepared",
    },
    "training_dataset": "data/processed/cleaned_cab_rides.csv",
    "rows_after_cleaning": 637976,
    "MAE": 1.4254,
    "RMSE": 2.6181,
    "R2_score": 0.9214,
    "CI_status": "PASS",
    "CT_status": "PASS with monitoring warning for service-name group error",
    "approval_status": "Approved for shadow deployment simulation",
    "warning_notes": [
        "Highest service-name error group is Lux Black XL.",
        "Use shadow deployment first and monitor group-level MAE before promotion.",
        "This model is a learning simulation, not a real production pricing system.",
    ],
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

registry = {"registered_models": [model_metadata]}
registry_path.write_text(json.dumps(registry, indent=2), encoding="utf-8")
print("Model registry saved to:", registry_path)
display(pd.DataFrame([model_metadata]))


Model registry saved to: d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops-tarif-transportasi-online\models\model_registry.json


,model_name,model_version,model_type,model_file,target_column,problem_type,features_used,features_excluded,training_dataset,rows_after_cleaning,MAE,RMSE,R2_score,CI_status,CT_status,approval_status,warning_notes,created_at
0,online_transport_fare_estimator,v1.0-baseline-random-forest,Random Forest Regressor,models/baseline_price_model.joblib,price,regression,"[distance, cab_type, source, destination, name...","{'id': 'identifier', 'product_id': 'identifier...",data/processed/cleaned_cab_rides.csv,637976,1.4254,2.6181,0.9214,PASS,PASS with monitoring warning for service-name ...,Approved for shadow deployment simulation,[Highest service-name error group is Lux Black...,2026-06-13T18:34:33


# Step 3: API Plan and Local FastAPI Skeleton

The trained model can be exposed through an API. The API receives trip information as input and returns an estimated fare as output.

This API is only a local demonstration for the MLOps pipeline. It uses the same feature format as the training pipeline.

Input features for the main endpoint:

- `distance`
- `cab_type`
- `source`
- `destination`
- `name`
- `time_stamp`

The API creates time features internally using the same preprocessing logic. It does not require `price` as input because `price` is the prediction target. It also does not use `surge_multiplier` in the main endpoint.


In [4]:
app_code = """from pathlib import Path
import sys

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from preprocessing import create_time_features  # noqa: E402


MODEL_VERSION = "v1.0-baseline-random-forest"
MODEL_PATH = PROJECT_ROOT / "models" / "baseline_price_model.joblib"
FEATURE_COLUMNS = [
    "distance",
    "cab_type",
    "source",
    "destination",
    "name",
    "hour",
    "day",
    "month",
    "day_of_week",
]


app = FastAPI(
    title="Online Transportation Fare Estimation API",
    description="Local learning simulation API for an MLOps mini project.",
    version=MODEL_VERSION,
)

model = joblib.load(MODEL_PATH)


class FarePredictionRequest(BaseModel):
    distance: float
    cab_type: str
    source: str
    destination: str
    name: str
    time_stamp: int


def validate_request(data):
    if data.distance <= 0:
        raise HTTPException(status_code=400, detail="distance must be greater than 0.")

    text_fields = {
        "cab_type": data.cab_type,
        "source": data.source,
        "destination": data.destination,
        "name": data.name,
    }
    for field_name, value in text_fields.items():
        if not value or not value.strip():
            raise HTTPException(status_code=400, detail=f"{field_name} must not be empty.")

    converted_time = pd.to_datetime(data.time_stamp, unit="ms", errors="coerce")
    if pd.isna(converted_time):
        raise HTTPException(status_code=400, detail="time_stamp must be a valid millisecond timestamp.")


@app.get("/")
def read_root():
    return {
        "message": "Online transportation fare estimation API is running.",
        "model_version": MODEL_VERSION,
        "note": "This is a local learning simulation, not a real production pricing system.",
    }


@app.post("/predict")
def predict_price(request: FarePredictionRequest):
    validate_request(request)

    if hasattr(request, "model_dump"):
        request_data = request.model_dump()
    else:
        request_data = request.dict()

    input_df = pd.DataFrame([request_data])
    input_df = create_time_features(input_df)
    prediction_input = input_df[FEATURE_COLUMNS]
    estimated_price = float(model.predict(prediction_input)[0])

    return {
        "estimated_price": round(estimated_price, 2),
        "model_version": MODEL_VERSION,
        "note": "This is a learning simulation for fare estimation, not a real production pricing system.",
    }
"""

request_example = {
    "distance": 2.5,
    "cab_type": "Uber",
    "source": "Back Bay",
    "destination": "North End",
    "name": "UberX",
    "time_stamp": 1544952607890,
}

api_readme = """# Local Fare Estimation API

This folder contains a simple FastAPI skeleton for the MLOps mini project. It is only a local learning simulation and is not a real production pricing system.

## Install Dependencies

```bash
pip install fastapi uvicorn joblib pandas scikit-learn
```

## Run Locally

From the project root, run:

```bash
uvicorn api.app:app --reload
```

Then open:

```text
http://127.0.0.1:8000/
```

## Example Request

Send a `POST` request to:

```text
http://127.0.0.1:8000/predict
```

Example body:

```json
{
  "distance": 2.5,
  "cab_type": "Uber",
  "source": "Back Bay",
  "destination": "North End",
  "name": "UberX",
  "time_stamp": 1544952607890
}
```

The same example is saved in `api/request_example.json`.

## Example Response

```json
{
  "estimated_price": 10.75,
  "model_version": "v1.0-baseline-random-forest",
  "note": "This is a learning simulation for fare estimation, not a real production pricing system."
}
```

The exact estimated price can be different depending on the saved model artifact.

## Limitations

- This API is a local demonstration only.
- It does not deploy to cloud services.
- It does not use `surge_multiplier` in the main endpoint.
- It does not require `price` as input because `price` is the prediction target.
- It does not merge weather data yet.
- It should not be used to determine real prices for any transportation platform.
"""

(api_dir / "app.py").write_text(app_code, encoding="utf-8")
(api_dir / "request_example.json").write_text(json.dumps(request_example, indent=2), encoding="utf-8")
(api_dir / "README.md").write_text(api_readme, encoding="utf-8")

api_summary = pd.DataFrame({
    "API File": ["api/app.py", "api/request_example.json", "api/README.md"],
    "Created": [
        (api_dir / "app.py").exists(),
        (api_dir / "request_example.json").exists(),
        (api_dir / "README.md").exists(),
    ],
    "Purpose": [
        "Local FastAPI skeleton",
        "Example request body",
        "Local run instructions",
    ],
})

display(api_summary)


,API File,Created,Purpose
0,api/app.py,True,Local FastAPI skeleton
1,api/request_example.json,True,Example request body
2,api/README.md,True,Local run instructions


# Step 4: Shadow Deployment Scenario

The selected deployment strategy is **Shadow Deployment**.

Shadow Deployment is suitable because:

- It is safer for a pricing-related model.
- Predictions can be compared with existing baseline outputs.
- Users are not affected during testing.
- Errors can be monitored before full release.

Simulated release flow:

1. Model v1.0 is trained and evaluated.
2. CI checklist validates data and preprocessing.
3. CT checklist validates performance and group-level errors.
4. Model is registered in `model_registry.json`.
5. Model is deployed as a local API simulation.
6. Shadow mode collects predictions in the background.
7. Monitoring checks MAE, RMSE, R2, and group errors.
8. If stable, model can be promoted to active simulation.
9. If unstable, rollback to previous approved model.


In [5]:
cd_checklist = pd.DataFrame([
    {"CD Component": "Model format", "Design Choice": ".joblib", "Purpose": "Store the trained sklearn pipeline", "Status": "READY" if model_file_available else "MISSING", "Notes": "Saved as models/baseline_price_model.joblib"},
    {"CD Component": "Model registry", "Design Choice": "models/model_registry.json", "Purpose": "Track model version, metrics, approval, and warnings", "Status": "READY" if registry_path.exists() else "MISSING", "Notes": "Simple JSON registry for mini project"},
    {"CD Component": "API framework", "Design Choice": "FastAPI", "Purpose": "Expose local prediction endpoint", "Status": "PLANNED", "Notes": "Local demo only, not cloud deployment"},
    {"CD Component": "Prediction endpoint", "Design Choice": "/predict", "Purpose": "Return estimated fare from trip input", "Status": "READY", "Notes": "Does not require price or surge_multiplier"},
    {"CD Component": "Deployment strategy", "Design Choice": "Shadow Deployment", "Purpose": "Test predictions in background before promotion", "Status": "DESIGNED", "Notes": "Safer simulation for pricing-related model"},
    {"CD Component": "Rollback strategy", "Design Choice": "Previous approved model version", "Purpose": "Return to earlier model if monitoring is unstable", "Status": "DESIGNED", "Notes": "Current project has one registered baseline version"},
    {"CD Component": "Monitoring metrics", "Design Choice": "MAE, RMSE, R2, group-level MAE", "Purpose": "Check accuracy and stability after release simulation", "Status": "DESIGNED", "Notes": "Service-name error warning must be monitored"},
    {"CD Component": "Logging plan", "Design Choice": "Input, output, timestamp, model version", "Purpose": "Support future monitoring and debugging", "Status": "PLANNED", "Notes": "No logging service is created in this step"},
    {"CD Component": "Safety note", "Design Choice": "Learning simulation only", "Purpose": "Avoid claiming real production pricing use", "Status": "DOCUMENTED", "Notes": "Not a real pricing system"},
])

display(cd_checklist)


,CD Component,Design Choice,Purpose,Status,Notes
0,Model format,.joblib,Store the trained sklearn pipeline,READY,Saved as models/baseline_price_model.joblib
1,Model registry,models/model_registry.json,"Track model version, metrics, approval, and wa...",READY,Simple JSON registry for mini project
2,API framework,FastAPI,Expose local prediction endpoint,PLANNED,"Local demo only, not cloud deployment"
3,Prediction endpoint,/predict,Return estimated fare from trip input,READY,Does not require price or surge_multiplier
4,Deployment strategy,Shadow Deployment,Test predictions in background before promotion,DESIGNED,Safer simulation for pricing-related model
5,Rollback strategy,Previous approved model version,Return to earlier model if monitoring is unstable,DESIGNED,Current project has one registered baseline ve...
6,Monitoring metrics,"MAE, RMSE, R2, group-level MAE",Check accuracy and stability after release sim...,DESIGNED,Service-name error warning must be monitored
7,Logging plan,"Input, output, timestamp, model version",Support future monitoring and debugging,PLANNED,No logging service is created in this step
8,Safety note,Learning simulation only,Avoid claiming real production pricing use,DOCUMENTED,Not a real pricing system


In [6]:
def dataframe_to_markdown(df):
    table_df = df.copy()
    headers = [str(column) for column in table_df.columns]
    rows = ["| " + " | ".join(headers) + " |"]
    rows.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for _, row in table_df.iterrows():
        rows.append("| " + " | ".join(str(value) for value in row.tolist()) + " |")
    return "\n".join(rows)

cd_checklist_markdown = dataframe_to_markdown(cd_checklist)

cd_report = f"""# CD Scenario Report

## Project Title

Mini MLOps Pipeline Design for Online Transportation Fare Estimation Based on Trip and Weather Data

## Step Title

CD Scenario, Model Registry, API Plan, and MLOps Pipeline Diagram

## CD Meaning in This Project

CD means preparing the trained model so it can be safely released as a prediction service. This step does not deploy anything to real cloud services. It only designs a local release scenario for learning.

## Deployment Strategy

The selected strategy is **Shadow Deployment**. In shadow mode, the new model makes predictions in the background first. Users are not affected during the test period.

## Simulated Release Flow

1. Model v1.0 is trained and evaluated.
2. CI checklist validates data and preprocessing.
3. CT checklist validates performance and group-level errors.
4. Model is registered in `models/model_registry.json`.
5. Model is wrapped in a local FastAPI skeleton.
6. Shadow mode collects predictions in the background.
7. Monitoring checks MAE, RMSE, R2, and group-level MAE.
8. If stable, the model can be promoted to active simulation.
9. If unstable, rollback to the previous approved model version.

## CD Checklist

{cd_checklist_markdown}

## Important Notes

- This model is a learning simulation for estimating online transportation fare.
- It is not a real production pricing system.
- Weather data is not merged yet.
- The model is not retrained in this step.
- No cloud deployment or real credentials are used.
"""

cd_report_path.write_text(cd_report, encoding="utf-8")
print("CD scenario report saved to:", cd_report_path)


CD scenario report saved to: d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops-tarif-transportasi-online\reports\cd_scenario_report.md


# Step 5: Final MLOps Pipeline Diagram

The pipeline connects data validation, preprocessing, training, evaluation, registry, API, deployment scenario, and monitoring.

The diagram is one of the final required outputs for the assignment. It should show CI, CT, and CD clearly.


In [7]:
mermaid_diagram = """```mermaid
flowchart LR
    subgraph DataLayer["Data Layer"]
        A["Raw Data<br/>cab_rides.csv + weather.csv"]
        B["Data Audit"]
    end

    subgraph CIStage["CI Stage"]
        C["CI Data Validation"]
        D["Data Cleaning"]
        E["Feature Engineering"]
        F["Preprocessing Pipeline"]
    end

    subgraph CTStage["CT Stage"]
        G["Model Training"]
        H["Model Evaluation"]
        I["CT Quality Checklist"]
    end

    subgraph CDStage["CD Stage"]
        J["Model Registry"]
        K["API Service"]
        L["Shadow Deployment"]
    end

    subgraph MonitoringStage["Monitoring Stage"]
        M["Monitoring"]
        N["Rollback or Promotion Decision"]
    end

    A --> B --> C --> D --> E --> F --> G --> H --> I --> J --> K --> L --> M --> N
    N -- "Promote if stable" --> K
    N -- "Rollback if unstable" --> J
```
"""

diagram_md_content = f"""# MLOps Pipeline Diagram

This Mermaid diagram shows the full mini MLOps pipeline for the online transportation fare estimation learning simulation.

{mermaid_diagram}

The diagram can be rendered in VSCode Markdown preview, GitHub Markdown, or Mermaid Live Editor.
"""

diagram_md_path.write_text(diagram_md_content, encoding="utf-8")
print("Mermaid diagram saved to:", diagram_md_path)


Mermaid diagram saved to: d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops-tarif-transportasi-online\diagrams\mlops_pipeline_diagram.md


In [8]:
def draw_pipeline_png(output_path):
    stages = [
        ("Data Layer", ["Raw Data", "Data Audit"]),
        ("CI Stage", ["CI Data\nValidation", "Data\nCleaning", "Feature\nEngineering", "Preprocessing\nPipeline"]),
        ("CT Stage", ["Model\nTraining", "Model\nEvaluation", "CT Quality\nChecklist"]),
        ("CD Stage", ["Model\nRegistry", "API\nService", "Shadow\nDeployment"]),
        ("Monitoring Stage", ["Monitoring", "Rollback or\nPromotion"]),
    ]

    fig, ax = plt.subplots(figsize=(18, 6))
    ax.set_xlim(0, 18)
    ax.set_ylim(0, 6)
    ax.axis("off")

    colors = ["#E8F4FA", "#EAF7EA", "#FFF4D8", "#F4EAFE", "#FDECEC"]
    box_color = "#FFFFFF"
    edge_color = "#2F3A45"

    x = 0.4
    y = 2.35
    box_w = 1.35
    box_h = 0.8
    gap = 0.28
    previous_center = None

    for stage_index, (stage_name, nodes) in enumerate(stages):
        stage_w = len(nodes) * box_w + (len(nodes) - 1) * gap + 0.35
        stage_x = x - 0.18
        ax.add_patch(Rectangle((stage_x, 1.45), stage_w, 3.2, facecolor=colors[stage_index], edgecolor="#B6C2CC", linewidth=1))
        ax.text(stage_x + stage_w / 2, 4.35, stage_name, ha="center", va="center", fontsize=11, fontweight="bold")

        for node in nodes:
            box = FancyBboxPatch(
                (x, y),
                box_w,
                box_h,
                boxstyle="round,pad=0.04,rounding_size=0.08",
                facecolor=box_color,
                edgecolor=edge_color,
                linewidth=1.2,
            )
            ax.add_patch(box)
            ax.text(x + box_w / 2, y + box_h / 2, node, ha="center", va="center", fontsize=9)

            current_center = (x + box_w / 2, y + box_h / 2)
            if previous_center is not None:
                ax.annotate(
                    "",
                    xy=(x, current_center[1]),
                    xytext=(previous_center[0] + box_w / 2, previous_center[1]),
                    arrowprops=dict(arrowstyle="->", color=edge_color, linewidth=1.2),
                )
            previous_center = current_center
            x += box_w + gap
        x += 0.35

    ax.text(9, 0.7, "CI validates data and preprocessing | CT validates model quality | CD prepares shadow deployment simulation", ha="center", fontsize=10)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

try:
    draw_pipeline_png(diagram_png_path)
    png_status = "created"
except Exception as exc:
    png_status = f"not created: {exc}"

print("PNG diagram status:", png_status)
print("PNG diagram path:", diagram_png_path)


PNG diagram status: created
PNG diagram path: d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops-tarif-transportasi-online\diagrams\mlops_pipeline_diagram.png


In [9]:
pipeline_design_report = f"""# MLOps Pipeline Design

## 1. Project Title

Mini MLOps Pipeline Design for Online Transportation Fare Estimation Based on Trip and Weather Data

## 2. Short Project Description

This project designs a mini MLOps pipeline for a learning simulation that estimates online transportation fare from trip data. The project uses simple, explainable stages so undergraduate students can present the workflow clearly.

## 3. Case and Model Selected

The selected case is fare estimation for online transportation trips. The selected model is the baseline Random Forest Regressor saved as `models/baseline_price_model.joblib`.

## 4. Dataset Used

- Raw trip data: `data/raw/cab_rides.csv`
- Raw weather data: `data/raw/weather.csv`
- Cleaned training data: `data/processed/cleaned_cab_rides.csv`

Weather data is not merged yet because time and location alignment must be planned carefully.

## 5. Target and Problem Type

- Target column: `price`
- Problem type: regression

## 6. CI Stage Explanation

The CI stage validates the data before training. It checks required columns, missing target values, positive distance, positive price, missing categorical values, valid timestamps, and duplicate rows.

## 7. CT Stage Explanation

The CT stage evaluates whether the trained model is accurate and stable. It monitors MAE, RMSE, R2 Score, and group-level MAE by distance group, cab type, and service name.

## 8. CD Stage Explanation

The CD stage prepares the model for a release simulation. In this project, CD creates a model registry, API skeleton, and shadow deployment plan. It does not deploy to cloud services.

## 9. Model Registry Explanation

The model registry is saved as `models/model_registry.json`. It tracks model version, model file path, metrics, CI/CT status, approval status, and warning notes.

## 10. API Plan Explanation

The API plan uses FastAPI locally. The API receives trip input fields and returns an estimated fare. The endpoint does not require `price` and does not use `surge_multiplier` for the main baseline.

## 11. Shadow Deployment Explanation

Shadow Deployment lets the model make predictions in the background first. This is safer because errors can be monitored before the model is promoted to active simulation.

## 12. Monitoring and Rollback Explanation

Monitoring should track MAE, RMSE, R2, group-level MAE, prediction inputs, prediction outputs, timestamp, and model version. If monitoring is unstable, the simulation should rollback to the previous approved model version.

## 13. Final Pipeline Diagram in Mermaid Format

{mermaid_diagram}

## 14. Notes for Presentation

- Explain that this is a learning simulation, not a real pricing system.
- Explain CI as data and preprocessing safety checks.
- Explain CT as model quality and stability checks.
- Explain CD as a local release design using registry, API plan, and shadow deployment.
- Mention that weather data is reserved for future improvement.
"""

pipeline_design_report_path.write_text(pipeline_design_report, encoding="utf-8")
print("MLOps pipeline design report saved to:", pipeline_design_report_path)


MLOps pipeline design report saved to: d:\Tugas Kuliah\SEM 6\Proyek Data Mining\mlops-tarif-transportasi-online\reports\mlops_pipeline_design.md


In [10]:
print("Final Step 4 Summary")
print("Model file checked:", model_file_available)
print("Model registry created:", registry_path.exists())
print("API skeleton created:", (api_dir / "app.py").exists())
print("CD scenario report created:", cd_report_path.exists())
print("Pipeline diagram created:", diagram_md_path.exists())
print("Pipeline PNG created:", diagram_png_path.exists())
print("Next step recommendation: prepare the final report and presentation slides.")


Final Step 4 Summary
Model file checked: True
Model registry created: True
API skeleton created: True
CD scenario report created: True
Pipeline diagram created: True
Pipeline PNG created: True
Next step recommendation: prepare the final report and presentation slides.
